In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# **EDA**

In [1]:
df = pd.read_csv('/content/T1.csv')

In [1]:
df.head()

Out[0]: 
          Date/Time  ...  Wind Direction (°)
0  01 01 2018 00:00  ...          259.994904
1  01 01 2018 00:10  ...          268.641113
2  01 01 2018 00:20  ...          272.564789
3  01 01 2018 00:30  ...          271.258087
4  01 01 2018 00:40  ...          265.674286

[5 rows x 5 columns]


          Date/Time  ...  Wind Direction (°)
0  01 01 2018 00:00  ...          259.994904
1  01 01 2018 00:10  ...          268.641113
2  01 01 2018 00:20  ...          272.564789
3  01 01 2018 00:30  ...          271.258087
4  01 01 2018 00:40  ...          265.674286

[5 rows x 5 columns]

In [1]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50530 entries, 0 to 50529
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Date/Time                      50530 non-null  object 
 1   LV ActivePower (kW)            50530 non-null  float64
 2   Wind Speed (m/s)               50530 non-null  float64
 3   Theoretical_Power_Curve (KWh)  50530 non-null  float64
 4   Wind Direction (°)             50530 non-null  float64
dtypes: float64(4), object(1)
memory usage: 1.9+ MB


This query filters the DataFrame to create df_cleaned, keeping only the "Farm is ON" data. It does this by selecting rows where the wind speed is high enough to generate power (> 3), the farm is actually producing power (> 1), and the output is physically possible (not > 0 theoretical power and not more than the theoretical maximum). This process removes all "shutdown" or "maintenance" events, which is what gives the model clean, consistent data to learn from.

In [1]:
df_cleaned = df.query(
    "`Wind Speed (m/s)` > 3 and "
    "`LV ActivePower (kW)` > 1 and "
    "`Theoretical_Power_Curve (KWh)` > 0 and "
    "`LV ActivePower (kW)` <= `Theoretical_Power_Curve (KWh)`"
)

In [1]:
df_cleaned.drop(['Date/Time','Theoretical_Power_Curve (KWh)'],axis=1,inplace=True)

<ipython-input-1-6c5f9b924025>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.drop(['Date/Time','Theoretical_Power_Curve (KWh)'],axis=1,inplace=True)


In [1]:
df_cleaned.head()

Out[0]: 
   LV ActivePower (kW)  Wind Speed (m/s)  Wind Direction (°)
0           380.047791          5.311336          259.994904
1           453.769196          5.672167          268.641113
2           306.376587          5.216037          272.564789
3           419.645905          5.659674          271.258087
4           380.650696          5.577941          265.674286


   LV ActivePower (kW)  Wind Speed (m/s)  Wind Direction (°)
0           380.047791          5.311336          259.994904
1           453.769196          5.672167          268.641113
2           306.376587          5.216037          272.564789
3           419.645905          5.659674          271.258087
4           380.650696          5.577941          265.674286

In [1]:
df_cleaned.rename(columns={'LV ActivePower (kW)': 'Power output'}, inplace=True)

<ipython-input-1-9b07b6d7156c>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.rename(columns={'LV ActivePower (kW)': 'Power output'}, inplace=True)


In [1]:
df_cleaned

Out[0]: 
       Power output  Wind Speed (m/s)  Wind Direction (°)
0        380.047791          5.311336          259.994904
1        453.769196          5.672167          268.641113
2        306.376587          5.216037          272.564789
3        419.645905          5.659674          271.258087
4        380.650696          5.577941          265.674286
...             ...               ...                 ...
50521   3333.819092         12.067660           81.985901
50522   3455.282959         12.195660           82.210617
50523   3429.021973         12.492510           82.111870
50524   3514.269043         12.559170           80.495262
50525   2963.980957         11.404030           80.502724

[32476 rows x 3 columns]


       Power output  Wind Speed (m/s)  Wind Direction (°)
0        380.047791          5.311336          259.994904
1        453.769196          5.672167          268.641113
2        306.376587          5.216037          272.564789
3        419.645905          5.659674          271.258087
4        380.650696          5.577941          265.674286
...             ...               ...                 ...
50521   3333.819092         12.067660           81.985901
50522   3455.282959         12.195660           82.210617
50523   3429.021973         12.492510           82.111870
50524   3514.269043         12.559170           80.495262
50525   2963.980957         11.404030           80.502724

[32476 rows x 3 columns]

In [1]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32476 entries, 0 to 50525
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Power output        32476 non-null  float64
 1   Wind Speed (m/s)    32476 non-null  float64
 2   Wind Direction (°)  32476 non-null  float64
dtypes: float64(3)
memory usage: 1014.9 KB


In [1]:
df_cleaned.duplicated().sum()

Out[0]: np.int64(0)


np.int64(0)

In [1]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(df_cleaned.columns), figsize=(15, 5))

for i, col in enumerate(df_cleaned.columns):
    # Convert to numeric, coercing errors to NaN
    numeric_col = pd.to_numeric(df_cleaned[col], errors='coerce')
    # Drop NaN values before plotting
    axes[i].boxplot(numeric_col.dropna())
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

<ipython-input-1-5797fc509965>:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# **creating the model**

In [1]:
from sklearn.model_selection import train_test_split


In [1]:
y = df_cleaned['Power output']
X = df_cleaned.drop('Power output', axis=1)

In [1]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

In [1]:
model = RandomForestRegressor(n_estimators=250, random_state=42, max_depth=12, min_samples_leaf=10)
model.fit(X_train, y_train)

from xgboost import XGBRegressor
xgb_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)


Out[0]: 
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [1]:
model_score = model.score(X_test, y_test)
xgb_score = xgb_model.score(X_test, y_test)
print(f"Random Forest R²: {model_score}")
print(f"XGBoost R²: {xgb_score}")


Random Forest R²: 0.958760872707207
XGBoost R²: 0.956708202358255


In [1]:
import joblib as jb

file_name = 'Power output model v1.joblib'

jb.dump(model, file_name)

Out[0]: ['Power output model v1.joblib']


['Power output model v1.joblib']

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/T1.csv')

df.head()

df.info()

df_cleaned = df.query(
    "`Wind Speed (m/s)` > 3 and "
    "`LV ActivePower (kW)` > 1 and "
    "`Theoretical_Power_Curve (KWh)` > 0 and "
    "`LV ActivePower (kW)` <= `Theoretical_Power_Curve (KWh)`"
)

df_cleaned.drop(['Date/Time','Theoretical_Power_Curve (KWh)'],axis=1,inplace=True)

df_cleaned.head()

df_cleaned.rename(columns={'LV ActivePower (kW)': 'Power output'}, inplace=True)

df_cleaned

df_cleaned.info()

df_cleaned.duplicated().sum()

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(df_cleaned.columns), figsize=(15, 5))

for i, col in enumerate(df_cleaned.columns):
    # Convert to numeric, coercing errors to NaN
    numeric_col = pd.to_numeric(df_cleaned[col], errors='coerce')
    # Drop NaN values before plotting
    axes[i].boxplot(numeric_col.dropna())
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

from sklearn.model_selection import train_test_split

y = df_cleaned['Power output']
X = df_cleaned.drop('Power output', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

model = RandomForestRegressor(n_estimators=250, random_state=42, max_depth=12, min_samples_leaf=10)
model.fit(X_train, y_train)

model_score = model.score(X_test, y_test)
print(f"R-squared score: {model_score}")

import joblib as jb

file_name = 'Power output model v1.joblib'

jb.dump(model, file_name)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50530 entries, 0 to 50529
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Date/Time                      50530 non-null  object 
 1   LV ActivePower (kW)            50530 non-null  float64
 2   Wind Speed (m/s)               50530 non-null  float64
 3   Theoretical_Power_Curve (KWh)  50530 non-null  float64
 4   Wind Direction (°)             50530 non-null  float64
dtypes: float64(4), object(1)
memory usage: 1.9+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 32476 entries, 0 to 50525
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Power output        32476 non-null  float64
 1   Wind Speed (m/s)    32476 non-null  float64
 2   Wind Direction (°)  32476 non-null  float64
dtypes: float64(3)
memory usage: 1014.9 KB
R-squared score: 0.9587608727

<ipython-input-1-ec6d0892b0c4>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.drop(['Date/Time','Theoretical_Power_Curve (KWh)'],axis=1,inplace=True)
<ipython-input-1-ec6d0892b0c4>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.rename(columns={'LV ActivePower (kW)': 'Power output'}, inplace=True)
<ipython-input-1-ec6d0892b0c4>:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


['Power output model v1.joblib']

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib as jb
import os

df = pd.read_csv('T1.csv')

df_cleaned = df.query(
    "`Wind Speed (m/s)` > 3 and "
    "`LV ActivePower (kW)` > 1 and "
    "`Theoretical_Power_Curve (KWh)` > 0 and "
    "`LV ActivePower (kW)` <= `Theoretical_Power_Curve (KWh)`"
)

df_cleaned = df_cleaned.drop(['Date/Time', 'Theoretical_Power_Curve (KWh)'], axis=1)
df_cleaned.rename(columns={'LV ActivePower (kW)': 'Power output'}, inplace=True)

print("Applying feature engineering for 'Wind Direction'...")
df_cleaned['wind_dir_sin'] = np.sin(np.radians(df_cleaned['Wind Direction (°)']))
df_cleaned['wind_dir_cos'] = np.cos(np.radians(df_cleaned['Wind Direction (°)']))

df_cleaned = df_cleaned.drop('Wind Direction (°)', axis=1)

print("Generating box plots for cleaned data...")
if not os.path.exists('eda_plots'):
    os.makedirs('eda_plots')

fig, axes = plt.subplots(1, len(df_cleaned.columns), figsize=(15, 5))
for i, col in enumerate(df_cleaned.columns):
    axes[i].boxplot(df_cleaned[col].dropna())
    axes[i].set_title(col)

plt.tight_layout()
plt.savefig('eda_plots/final_boxplots.png')
plt.show()

y = df_cleaned['Power output']
X = df_cleaned.drop('Power output', axis=1)

print("\nFinal features being used to train the model:")
X.info()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model...")
model = RandomForestRegressor(n_estimators=250, random_state=42, max_depth=12, min_samples_leaf=10)
model.fit(X_train, y_train)

model_score = model.score(X_test, y_test)
print(f"R-squared score: {model_score}")

file_name = 'Power output model v1.joblib'
jb.dump(model, file_name)
print(f"Model saved as '{file_name}'")


Applying feature engineering for 'Wind Direction'...
Generating box plots for cleaned data...

Final features being used to train the model:
<class 'pandas.core.frame.DataFrame'>
Index: 32476 entries, 0 to 50525
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Wind Speed (m/s)  32476 non-null  float64
 1   wind_dir_sin      32476 non-null  float64
 2   wind_dir_cos      32476 non-null  float64
dtypes: float64(3)
memory usage: 1014.9 KB
Training model...
R-squared score: 0.9586629181761483
Model saved as 'Power output model v1.joblib'


<ipython-input-1-806147f57ec0>:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib as jb
import os

df = pd.read_csv('/content/T1.csv')


df_cleaned = df.query(
    "`Wind Speed (m/s)` > 3 and "
    "`LV ActivePower (kW)` > 1 and "
    "`Theoretical_Power_Curve (KWh)` > 0 and "
    "`LV ActivePower (kW)` <= `Theoretical_Power_Curve (KWh)`"
)

df_cleaned = df_cleaned.drop(['Date/Time', 'Theoretical_Power_Curve (KWh)'], axis=1)
df_cleaned.rename(columns={'LV ActivePower (kW)': 'Power output'}, inplace=True)


df_cleaned['wind_dir_sin'] = np.sin(np.radians(df_cleaned['Wind Direction (°)']))
df_cleaned['wind_dir_cos'] = np.cos(np.radians(df_cleaned['Wind Direction (°)']))
df_cleaned = df_cleaned.drop('Wind Direction (°)', axis=1)


if not os.path.exists('eda_plots'):
    os.makedirs('eda_plots')

fig, axes = plt.subplots(1, len(df_cleaned.columns), figsize=(15, 5))
for i, col in enumerate(df_cleaned.columns):
    axes[i].boxplot(df_cleaned[col].dropna())
    axes[i].set_title(col)

plt.tight_layout()
plt.savefig('eda_plots/final_boxplots.png')
plt.show()

y = df_cleaned['Power output']
X = df_cleaned.drop('Power output', axis=1)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=250, random_state=42, max_depth=12, min_samples_leaf=10)
model.fit(X_train, y_train)

model_score = model.score(X_test, y_test)
print(f"R-squared score: {model_score}")

file_name = 'Power output model v1.joblib'
jb.dump(model, file_name)
print(f"Model saved as '{file_name}'")

R-squared score: 0.9586629181761483
Model saved as 'Power output model v1.joblib'


<ipython-input-1-bef505cdf751>:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
